# Preparação da base municipal do IDEB 2023–2025

Este notebook realiza somente a primeira etapa do projeto: leitura do arquivo bruto, seleção da rede municipal, padronização dos valores ausentes, validações e exportação da base supervisionada.

**Regra temporal:** `VL_OBSERVADO_2023` é uma variável histórica permitida; `VL_OBSERVADO_2025` é exclusivamente o alvo. Nenhuma variável posterior a 2024 deve entrar como preditora.


## 1. Configuração e carregamento da função de preparação

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preparar_ideb import prepare_base

print(f"Projeto: {PROJECT_ROOT}")


## 2. Ler, filtrar e limpar o arquivo do IDEB

In [ ]:
result = prepare_base(PROJECT_ROOT)
raw = result["raw"]
municipal = result["municipal"]
base = result["base"]

result["summary"]


## 3. Verificações de qualidade

In [ ]:
checks = {
    "somente_rede_municipal": municipal["REDE"].dropna().eq("Municipal").all(),
    "codigo_municipio_unico": municipal["CO_MUNICIPIO"].is_unique,
    "codigo_com_7_digitos": municipal["CO_MUNICIPIO"].str.len().eq(7).all(),
    "sem_alvo_ausente": base["VL_OBSERVADO_2025"].notna().all(),
    "sem_historico_ausente": base["VL_OBSERVADO_2023"].notna().all(),
    "tamanho_intersecao_esperado": len(base) == 5196,
}
checks


In [ ]:
preview = base[
    ["SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO", "REDE", "VL_OBSERVADO_2023", "VL_OBSERVADO_2025"]
].head(10)
preview


## 4. Visualização inicial do alvo e da variável histórica

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(base["VL_OBSERVADO_2025"], bins=20, color="#2F6690", edgecolor="white")
axes[0].set_title("Distribuição do IDEB 2025")
axes[0].set_xlabel("IDEB observado")
axes[0].set_ylabel("Quantidade de municípios")

axes[1].scatter(
    base["VL_OBSERVADO_2023"],
    base["VL_OBSERVADO_2025"],
    s=12,
    alpha=0.45,
    color="#D1495B",
)
axes[1].set_title("IDEB 2023 versus IDEB 2025")
axes[1].set_xlabel("IDEB 2023")
axes[1].set_ylabel("IDEB 2025")

plt.tight_layout()
plt.show()


## 5. Arquivos gerados

O notebook salva:

- `data/processed/ideb_municipal_2023_2025.csv`: base limpa com os municípios que possuem IDEB 2023 e 2025;
- `results/ideb_municipal_2023_2025_resumo.csv`: contagens de controle da preparação.

O próximo notebook deverá integrar essa base com os Indicadores Educacionais, o Censo Escolar e o IBGE.
